# Step 1c – ভুল করে বাদ পড়া রেকর্ডিং ফেরত আনা (GPU লাগবে না)

**Runtime → Run all** চাপুন। CPU-তেই চলবে (GPU নেবেন না, ইউনিট বাঁচবে)। ~৫–১০ মিনিট।

**কেন:** Step 1b-এর একটা পুরনো ভার্সনে যেসব রেকর্ডিংয়ের লেখা Whisper ঠিকমতো বের করতে পারেনি (`check_ear`, যেমন বাংলার বদলে ইংরেজি/হিন্দি অক্ষরে লিখেছে), সেগুলোও বাদের তালিকায় চলে গিয়েছিল। এই নোটবুক Drive-এ সেভ থাকা Whisper-এর লেখাগুলো নতুন নিয়মে আবার মেলায়:
- **ok / uncertain / check_ear** → ট্রেনিংয়ে থাকবে
- **moved / shift** নিশ্চিত → সঠিক বাক্যে সরানো হবে
- **moved / shift** অনিশ্চিত, বা ডুপ্লিকেট → বাদ

ব্যবহৃত বক্তা: A1, A2, A4, A6, A7। পুরনো ফাইলগুলো মুছবে না, `recheck/backup_...` ফোল্ডারে কপি থাকবে।
শেষে নতুন `metadata.csv` তৈরি হবে; Step 2d / 2e কিছু না বদলেই সেটা পড়বে।

## 1. Drive আর সেটিংস

In [ ]:
from google.colab import drive
import os
if not os.path.ismount('/content/drive') and os.path.isdir('/content/drive'):
    for d, _, _ in sorted(os.walk('/content/drive'), key=lambda x: -len(x[0])):
        try: os.rmdir(d)
        except OSError: pass
drive.mount('/content/drive')

In [ ]:
import os, pandas as pd, numpy as np, shutil, datetime, subprocess, sys
def first(*paths):
    for p in paths:
        if os.path.exists(p): return p
    raise FileNotFoundError(paths[0])
ROOT = first("/content/drive/MyDrive/Sylheti_Thesis/Ultimate Thisis Data", "/content/drive/MyDrive/Ultimate Thisis Data")
ASR_DIR = first("/content/drive/MyDrive/Sylheti_Thesis/sylheti_asr", "/content/drive/MyDrive/sylheti_asr")
SAVE = ROOT + "/wav_16k_tar"
RC = ROOT + "/recheck"
OUT = "/content/wav_16k"                     # audio paths written into metadata (same as Step 2b/2d)
TRAIN_SPK = ["A1", "A4", "A6", "A7"]
TEST_SPEAKER = "A2"
KEEP = TRAIN_SPK + [TEST_SPEAKER]
STATUSES = ["primary", "second_voice", "alt_take", "unknown"]
for f in [ROOT + "/mapping.csv", ROOT + "/corpus.csv", SAVE + "/metadata.csv", RC + "/recheck_lib_v2.py"] + [f"{RC}/transcripts_{s}.csv" for s in KEEP]:
    assert os.path.exists(f), "পাওয়া যায়নি: " + f
print("ROOT   :", ROOT); print("ASR_DIR:", ASR_DIR)

## 2. মেলানোর লাইব্রেরি (Step 1b-এর নতুন ভার্সন)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz"], check=True)

In [ ]:
shutil.copy(RC + "/recheck_lib_v2.py", "/content/recheck_lib.py")   # same matching code as the newest Step 1b
print("recheck_lib ready")

## 3. সেভ থাকা Whisper-এর লেখা দিয়ে আবার মেলানো

In [ ]:
sys.path.insert(0, "/content")
import importlib, recheck_lib; importlib.reload(recheck_lib)
from recheck_lib import match, summary
m = pd.read_csv(ROOT + "/mapping.csv", encoding="utf-8-sig")
m = m[m.status.isin(STATUSES)].reset_index(drop=True)
m["order"] = np.arange(len(m))                      # recording order (same as Step 1b)
m["excel_row"] = m.excel_row.fillna(-1).astype(int)
m["id"] = m.original_file
m = m[m.speaker.isin(KEEP)].reset_index(drop=True)
corpus = pd.read_csv(ROOT + "/corpus.csv", encoding="utf-8-sig")
tr = pd.concat([pd.read_csv(f"{RC}/transcripts_{s}.csv", encoding="utf-8-sig") for s in KEEP])
tr["asr"] = tr.asr.fillna("")
x = m.merge(tr, on="id", how="inner")
print("মেলানো হচ্ছে:", len(x), "টা রেকর্ডিং (কয়েক মিনিট)")
res = match(x[["wav_name", "speaker", "excel_row", "order", "asr", "status", "original_file"]], corpus)
txt = corpus.set_index("excel_row")
res["filed_sylheti"] = res.excel_row.map(txt.sylheti); res["suggested_sylheti"] = res.suggested_row.map(txt.sylheti)
sm = summary(res)
print(sm.to_string())

## 4. পুরনো ফাইলের ব্যাকআপ, তারপর নতুন তালিকা

In [ ]:
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
BK = f"{RC}/backup_{stamp}"
os.makedirs(BK, exist_ok=True)
for f in [RC + "/drop.csv", RC + "/corrections.csv", RC + "/report.csv", RC + "/check_by_ear.csv", RC + "/summary.csv",
          SAVE + "/metadata_splits.csv", ASR_DIR + "/metadata.csv"]:
    if os.path.exists(f):
        shutil.copy2(f, f"{BK}/{os.path.basename(os.path.dirname(f))}__{os.path.basename(f)}")
print("ব্যাকআপ:", BK, os.listdir(BK))
old_meta = pd.read_csv(f"{BK}/{os.path.basename(ASR_DIR)}__metadata.csv") if os.path.exists(f"{BK}/{os.path.basename(ASR_DIR)}__metadata.csv") else None

cols = ["speaker", "status", "original_file", "wav_name", "excel_row", "verdict", "suggested_row", "auto_fix",
        "score_filed", "score_best", "asr", "filed_sylheti", "suggested_sylheti"]
res = res[cols].sort_values(["speaker", "excel_row"])
res.to_csv(RC + "/report.csv", index=False, encoding="utf-8-sig")
fix = res[res.auto_fix & res.wav_name.notna()]
fix[["wav_name", "speaker", "excel_row", "suggested_row", "verdict", "score_filed", "score_best", "asr",
     "filed_sylheti", "suggested_sylheti"]].to_csv(RC + "/corrections.csv", index=False, encoding="utf-8-sig")
ear = res[(res.verdict != "ok") & ~res.auto_fix]
ear.to_csv(RC + "/check_by_ear.csv", index=False, encoding="utf-8-sig")
# only clips that clearly say ANOTHER sentence but could not be moved safely are left out.
# check_ear (Whisper could not write it) and uncertain clips stay in training.
drop = ear[ear.wav_name.notna() & ear.verdict.str.startswith(("moved", "shift"))][["wav_name", "verdict", "asr"]]
drop.to_csv(RC + "/drop.csv", index=False, encoding="utf-8-sig")
sm.to_csv(RC + "/summary.csv", encoding="utf-8-sig")
print(f"ঠিক বাক্যে সরানো (corrections.csv): {len(fix)}")
print(f"বাদ (drop.csv): {len(drop)}   | আগে ছিল: {len(pd.read_csv(BK + '/recheck__drop.csv')) if os.path.exists(BK + '/recheck__drop.csv') else '-'}")
print(drop.verdict.value_counts().to_string())

## 5. নতুন train / validation / test তালিকা (Step 1b-এর শেষ সেলের মতো)

In [ ]:
meta = pd.read_csv(SAVE + "/metadata.csv")
meta = meta[meta.speaker.isin(KEEP)]
bad = set(pd.read_csv(SAVE + "/conversion_report.csv", encoding="utf-8-sig").get("original_file", [])) \
      if os.path.exists(SAVE + "/conversion_report.csv") else set()
fx = fix.set_index("wav_name").suggested_row
hit = meta.file_name.isin(fx.index)
meta.loc[hit, "excel_row"] = meta.loc[hit, "file_name"].map(fx)
print("corrections applied:", int(hit.sum()))
meta = meta[~meta.file_name.isin(set(drop.wav_name))]
mm = pd.read_csv(ROOT + "/mapping.csv", encoding="utf-8-sig")
meta = meta[~meta.file_name.isin(set(mm[mm.original_file.isin(bad)].wav_name))]
meta = meta.drop(columns=[c for c in ["sylheti", "bangla", "english", "split", "text"] if c in meta.columns]).merge(
    corpus[["excel_row", "split", "sylheti_asr", "sylheti", "bangla", "english"]], on="excel_row", how="left")
held = meta.speaker == TEST_SPEAKER
meta["split"] = meta.split.map({"train": "train", "val": "validation", "test": "test_seen"})
meta.loc[held, "split"] = meta.loc[held, "split"].map({"test_seen": "test_unseen"}).fillna("unused")
meta = meta.rename(columns={"sylheti_asr": "text"})
meta.to_csv(SAVE + "/metadata_splits.csv", index=False)
asr = meta[meta.split != "unused"].assign(audio_path=OUT + "/" + meta.file_name)
asr = asr[["audio_path", "text", "split", "speaker", "excel_row", "sylheti", "bangla", "english", "duration_sec"]]
asr.to_csv(ASR_DIR + "/metadata.csv", index=False)

new_t = asr.groupby(["speaker", "split"]).size().unstack(fill_value=0)
print("\nনতুন তালিকা (রেকর্ডিং):"); print(new_t.to_string())
print("\nঘণ্টা:"); print(asr.groupby("split").duration_sec.sum().div(3600).round(2).to_string())
print("\nমোট:", len(asr), "রেকর্ডিং,", round(asr.duration_sec.sum() / 3600, 2), "ঘণ্টা")
if old_meta is not None:
    o = old_meta[old_meta.speaker.isin(KEEP)]
    cmp = pd.DataFrame({"আগে": o.groupby("speaker").size(), "এখন": asr.groupby("speaker").size()}).fillna(0).astype(int)
    cmp["বেড়েছে"] = cmp["এখন"] - cmp["আগে"]
    print("\nআগে বনাম এখন:"); print(cmp.to_string())
print("\nsaved:", ASR_DIR + "/metadata.csv")

## 6. (ঐচ্ছিক) ফেরত আসা কয়েকটা রেকর্ডিং শুনে দেখা

In [ ]:
from IPython.display import Audio, display
back = res[(res.verdict == "check_ear") & res.wav_name.notna()].sample(5, random_state=7)
for _, r in back.iterrows():
    src = f"{ROOT}/Thesis Data 2/" + str(r.original_file).replace("\\", "/")
    raw = subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", src, "-ac", "1", "-ar", "16000", "-f", "f32le", "-"],
                         capture_output=True, timeout=120).stdout
    a = np.frombuffer(raw, dtype=np.float32)
    print(f"{r.speaker}  বাক্য {r.excel_row}:", r.filed_sylheti); print("   Whisper:", r.asr)
    if len(a): display(Audio(a, rate=16000))
    print()

---
শেষ হলে ৩ আর ৫ নম্বর সেকশনের আউটপুটের স্ক্রিনশট দিন। তারপর Step 2d (turbo) আর 2e (small) `QUICK = False` দিয়ে চালাবেন।